# XGBoost Survival Embeddings (XGBSE) - Training Pipeline

Questo notebook definisce il processo di addestramento per il modello basato su gradient boosting applicato all'analisi di sopravvivenza. Nello specifico, implementa l'algoritmo XGBSEDebiasedBCE della libreria XGBSE al posto del tradizionale XGBoost Cox.

---

### Fase 1: Caricamento e Preprocessing (Selezione Feature)
Il dataset viene caricato e i valori mancanti originari (`-1`) vengono convertiti in `NaN`. A differenza della foresta casuale, per questo modello filtriamo il dataset utilizzando un sottoinsieme predefinito di 13 feature clinicamente rilevanti (es. `Final_N_enc`, `Intent_CURE`, `pack_years_smoked`). Questo riduce la dimensionalità e ottimizza la convergenza dei gradienti.

### Fase 2: Train / Test Split
Il dataset viene suddiviso in set di addestramento (75%) e test (25%).
La suddivisione è stratificata sull'esito (`vital_status_DEAD`) per mantenere costante il tasso di censura e di eventi osservati in entrambi i sottoinsiemi, garantendo una valutazione non distorta.

### Fase 3: Configurazione Iperparametri e Scelte Architetturali
Viene definita una griglia di iperparametri per la ricerca randomizzata.
Una scelta tecnica rilevante è l'impostazione di `tree_method="exact"`. Dato che il dataset Head & Neck Cancer ha dimensioni contenute, possiamo permetterci di utilizzare l'algoritmo esatto per la divisione dei nodi, che garantisce una matematica più precisa rispetto alle approssimazioni basate su istogrammi usate per i big data.

### Fase 4: Ottimizzazione e Addestramento (Perché XGBSEDebiasedBCE?)
Il modello viene ottimizzato tramite `RandomizedSearchCV` con 150 iterazioni e 5-fold CV.
La scelta di **XGBSEDebiasedBCE** al posto dell'obiettivo "survival:cox" standard è strategica: questo approccio predice la funzione di sopravvivenza direttamente addestrando un set di classificatori logistici nel tempo (Binary Cross Entropy). Questo garantisce una calibrazione nativa superiore e un Integrated Brier Score (IBS) migliore, eliminando la necessità di ricorrere a stimatori post-hoc imprecisi come l'approssimazione di Breslow.

### Fase 5: Valutazione Rapida e Salvataggio dell'Artefatto
Dopo l'addestramento, calcoliamo il C-index sui dati di training e test.
Per il salvataggio, invece di esportare solo il modello, creiamo un "artefatto" completo (`pickle` dictionary) che include il modello, la lista delle feature e i dati di addestramento grezzi. Questo passaggio è fondamentale per fornire a SHAP il corretto dataset di "background" durante la fase di Explainable AI nel notebook di valutazione.

In [20]:
import os, pickle, warnings, random
import numpy as np
import pandas as pd
import joblib
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored
from xgbse import XGBSEDebiasedBCE
from sklearn.model_selection import train_test_split, RandomizedSearchCV

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURAZIONE E PATHS
# ==============================================================================
def get_project_root(current_path):
    if os.path.exists(os.path.join(current_path, "data", "dataset_cleaned.csv")):
        return current_path
    parent = os.path.dirname(current_path)
    if parent == current_path:
        raise FileNotFoundError("Root del progetto non trovata.")
    return get_project_root(parent)

CWD = os.getcwd()
PROJECT_ROOT = get_project_root(CWD)

DATA_PATH  = os.path.join(PROJECT_ROOT, "data", "dataset_cleaned.csv")
MODEL_DIR  = os.path.join(PROJECT_ROOT, "models")
MODEL_PATH = os.path.join(MODEL_DIR, "xgbse_model.pkl")

os.makedirs(MODEL_DIR, exist_ok=True)

TIME_COL     = "follow_ups.days_to_follow_up"
EVENT_COL    = "demographic.vital_status_DEAD"
RANDOM_STATE = 20
TEST_SIZE    = 0.25
N_ITER       = 150

FEATURE_COLS = [
    "Final_N_enc",
    "Intent_CURE",
    "Hist_RECURRENCE",
    "demographic.age_at_index",
    "Hist_METASTASIS",
    "demographic.ethnicity_HISPANIC OR LATINO",
    "Trt_RADIATION",
    "exposures.pack_years_smoked",
    "diagnoses.laterality_RIGHT",
    "diagnoses.laterality_MIDLINE",
    "demographic.race_NOT WHITE",
    "Trt_PHARMACEUTICAL",
    "exposures.alcohol_days_per_week",
]

# ==============================================================================
# FASE 1: CARICAMENTO E PREPROCESSING
# ==============================================================================
df = pd.read_csv(DATA_PATH)
df = df.replace(-1, np.nan)

X = df[FEATURE_COLS].copy()
y = Surv.from_arrays(event=df[EVENT_COL].astype(bool), time=df[TIME_COL])

# ==============================================================================
# FASE 2: SPLIT TRAIN / TEST
# ==============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df[EVENT_COL],
)

# ==============================================================================
# FASE 3: GRIGLIA IPERPARAMETRI E OTTIMIZZAZIONE
# ==============================================================================
rng = random.Random(RANDOM_STATE)
xgb_params_list = []

for _ in range(N_ITER):
    params = {
        "objective":        "survival:cox",
        "eval_metric":      "cox-nloglik",
        "tree_method":      "exact",
        "learning_rate":    rng.choice([0.005, 0.01, 0.03, 0.05]),
        "max_depth":        rng.choice([1, 2, 3]),
        "min_child_weight": rng.choice([5, 10, 15, 20]),
        "gamma":            rng.choice([0, 0.5, 1, 5]),
        "subsample":        rng.choice([0.5, 0.65, 0.8]),
        "colsample_bynode": rng.choice([0.2, 0.3, 0.5, 0.7]),
        "reg_alpha":        rng.choice([0.1, 1, 5, 10]),
        "reg_lambda":       rng.choice([1, 5, 10, 20]),
        "seed":             RANDOM_STATE,
    }
    xgb_params_list.append(params)

param_distributions = {"xgb_params": xgb_params_list}

def score_xgbse(estimator, X_val, y_val):
    p_df     = estimator.predict(X_val)
    r_scores = 1.0 - p_df.iloc[:, -1].values
    return concordance_index_censored(y_val["event"], y_val["time"], r_scores)[0]

search = RandomizedSearchCV(
    XGBSEDebiasedBCE(),
    param_distributions,
    n_iter=N_ITER,
    cv=5,
    scoring=score_xgbse,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0,
)
search.fit(X_train, y_train)

best_model  = search.best_estimator_
best_params = search.best_params_["xgb_params"]
best_cv_ci  = search.best_score_

# ==============================================================================
# FASE 4: VALUTAZIONE RAPIDA E SALVATAGGIO
# ==============================================================================
preds_test  = best_model.predict(X_test)
risk_scores = 1.0 - preds_test.iloc[:, -1].values
ci_test     = concordance_index_censored(y_test["event"], y_test["time"], risk_scores)[0]

preds_train = best_model.predict(X_train)
risk_train  = 1.0 - preds_train.iloc[:, -1].values
ci_train    = concordance_index_censored(y_train["event"], y_train["time"], risk_train)[0]

artifact = {
    "model":          best_model,
    "feature_cols":   FEATURE_COLS,
    "X_train_df":     X_train,
    "y_train":        y_train,
    "y_test":         y_test,
    "t_train":        np.array([s[1] for s in y_train]),
    "e_train":        np.array([s[0] for s in y_train]).astype(int),
    "t_test":         np.array([s[1] for s in y_test]),
    "e_test":         np.array([s[0] for s in y_test]).astype(int),
    "cv_c_index":     best_cv_ci,
    "ci_train":       ci_train,
    "ci_test":        ci_test,
    "best_params":    best_params,
    "random_state":   RANDOM_STATE,
    "test_size":      TEST_SIZE,
}

with open(MODEL_PATH, "wb") as f:
    pickle.dump(artifact, f)

joblib.dump(best_model, os.path.join(MODEL_DIR, "xgbse_model_joblib.pkl"))

print("=" * 62)
print("  TRAINING COMPLETATO - XGBSE (DebiasedBCE)")
print("=" * 62)
print(f"  CV C-index (best) : {best_cv_ci:.4f}")
print(f"  C-index TRAIN     : {ci_train:.4f}")
print(f"  C-index TEST      : {ci_test:.4f}")
print(f"  Artefatti salvati in: {MODEL_DIR}/")
print("=" * 62)

  TRAINING COMPLETATO - XGBSE (DebiasedBCE)
  CV C-index (best) : 0.6636
  C-index TRAIN     : 0.8523
  C-index TEST      : 0.7149
  Artefatti salvati in: C:\Users\nicon\PycharmProjects\ProgettoTesi\models/


# XGBoost Survival Embeddings (XGBSE) - Evaluation & Explainable AI

Questo notebook analizza le performance e le logiche decisionali del modello XGBSE precedentemente addestrato. L'obiettivo è quantificare l'accuratezza predittiva e utilizzare tecniche di Explainable AI (XAI) per ispezionare l'impatto delle singole feature sul rischio calcolato.

---

### Fase 1: Caricamento Artefatti e Ricostruzione Test Set
Il notebook inizia ricaricando l'artefatto salvato durante l'addestramento.
Vengono estratti il modello ottimizzato e i dati necessari, ricostruendo l'esatto Test Set attraverso lo stesso `random_state`. A differenza del classico XGBoost, XGBSE restituisce nativamente un dataframe dove le colonne rappresentano gli step temporali e i valori la probabilità di sopravvivenza $S(t)$. Da questo, deriviamo un "Risk Score" scalare utile per l'ordinamento dei pazienti (calcolato come $1 - S(t_{finale})$).

### Fase 2: Metriche Globali di Sopravvivenza
Valutiamo la performance attraverso le metriche standard:
* **C-index:** Misura l'abilità di ranking del modello.
* **Integrated Brier Score (IBS):** Valuta l'errore quadratico medio delle previsioni di sopravvivenza nel tempo. Grazie all'approccio DebiasedBCE, ci aspettiamo un IBS particolarmente basso (ben calibrato). I tempi predetti dal modello vengono interpolati dinamicamente per allinearsi agli standard di calcolo della libreria scikit-survival.
* **Time-dependent AUC:** Misura la capacità di discriminazione a specifici cut-off temporali (1, 3 e 5 anni).

### Fase 3: Stratificazione del Rischio (Validazione Clinica)
I pazienti del test set vengono classificati in "Alto Rischio" e "Basso Rischio" rispetto alla mediana dei risk score.
Avendo nativamente le probabilità per ogni step temporale, le curve di sopravvivenza per queste due coorti vengono ottenute calcolando direttamente la media delle funzioni di sopravvivenza predette, permettendo una validazione visiva della capacità di stratificazione del modello.

### Fase 4: Explainable AI Globale (Permutation Feature Importance)
Applichiamo la Permutation Feature Importance incapsulando il modello XGBSE in un wrapper personalizzato compatibile con scikit-learn.
La metrica di riferimento (C-index) viene ricalcolata mescolando ripetutamente i valori di una feature alla volta. Questo ci rivela in percentuale quali tra le 13 feature cliniche inizialmente selezionate sono i veri "driver" della predizione per la popolazione generale.

### Fase 5: Explainable AI Locale (SHAP)
Utilizziamo la libreria SHAP per fornire un'interpretazione paziente-centrica:
* **SHAP Summary Plot:** Mostra la distribuzione degli impatti (positivi o negativi sul punteggio di rischio) per ciascuna feature sull'intero set di test.
* **SHAP Waterfall Plot:** Generati per i pazienti con il rischio più alto e più basso della coorte. Dimostrano in modo granulare come il modello parte dal rischio base della popolazione e lo aggiorna matematicamente in base ai valori specifici di quel paziente (es. presenza di metastasi, anni di fumo), fornendo una spiegazione white-box clinica.

In [21]:
import os, pickle, warnings
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sksurv.util import Surv
from sksurv.metrics import (concordance_index_censored,
                             integrated_brier_score,
                             cumulative_dynamic_auc)

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURAZIONE E PATHS
# ==============================================================================
def get_project_root(current_path):
    if os.path.exists(os.path.join(current_path, "data", "dataset_cleaned.csv")):
        return current_path
    parent = os.path.dirname(current_path)
    if parent == current_path:
        raise FileNotFoundError("Root del progetto non trovata.")
    return get_project_root(parent)

CWD = os.getcwd()
PROJECT_ROOT = get_project_root(CWD)

if os.path.basename(CWD) == "XGBoostSurvival":
    SCRIPT_DIR = CWD
else:
    SCRIPT_DIR = os.path.join(PROJECT_ROOT, "XGBoostSurvival")

RESULTS_DIR = os.path.join(SCRIPT_DIR, "results")
DATA_PATH   = os.path.join(PROJECT_ROOT, "data", "dataset_cleaned.csv")
MODEL_DIR   = os.path.join(PROJECT_ROOT, "models")
MODEL_PATH  = os.path.join(MODEL_DIR, "xgbse_model.pkl")

os.makedirs(RESULTS_DIR, exist_ok=True)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Artefatto non trovato: {MODEL_PATH}. Esegui prima il training.")

TIME_COL  = "follow_ups.days_to_follow_up"
EVENT_COL = "demographic.vital_status_DEAD"

# ==============================================================================
# 1. CARICAMENTO ARTEFATTO E TEST SET
# ==============================================================================
print("=" * 62)
print("  EVALUATION & XAI - XGBSE (DebiasedBCE)")
print("=" * 62)

with open(MODEL_PATH, "rb") as f:
    art = pickle.load(f)

model        = art["model"]
feature_cols = art["feature_cols"]
X_train_df   = art["X_train_df"]
y_train      = art["y_train"]
t_train      = art["t_train"]
e_train      = art["e_train"]
t_test       = art["t_test"]
e_test       = art["e_test"]
RANDOM_STATE = art["random_state"]
TEST_SIZE    = art["test_size"]

df = pd.read_csv(DATA_PATH)
df = df.replace(-1, np.nan)

X_all = df[feature_cols].copy()
y_all = Surv.from_arrays(event=df[EVENT_COL].astype(bool), time=df[TIME_COL])

_, X_test, _, y_test_sksurv = train_test_split(
    X_all, y_all,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df[EVENT_COL],
)
y_test_sksurv = Surv.from_arrays(event=e_test.astype(bool), time=t_test)

preds_df    = model.predict(X_test)
risk_scores = 1.0 - preds_df.iloc[:, -1].values
preds_train_df = model.predict(X_train_df)

print(f"[+] Artefatto caricato | CV C-index: {art['cv_c_index']:.4f} | Feature: {len(feature_cols)}")
print(f"[+] Test set: {len(X_test)} pz | {e_test.sum()} eventi\n")

# ==============================================================================
# FASE 1: METRICHE GLOBALI
# ==============================================================================
print("=" * 62)
print("  FASE 1: METRICHE GLOBALI")
print("=" * 62)

c_index = concordance_index_censored(e_test.astype(bool), t_test, risk_scores)[0]

lb        = np.percentile(t_test, 10)
ub        = np.percentile(t_test, 90)
ibs_times = np.linspace(lb, ub, 20)

preds_interp = np.clip(
    np.array([
        np.interp(ibs_times, preds_df.columns.astype(float), preds_df.iloc[i].values)
        for i in range(len(preds_df))
    ]),
    1e-6, 1 - 1e-6,
)

ibs = integrated_brier_score(y_train, y_test_sksurv, preds_interp, ibs_times)

tempi_giorni = [365, 1095, 1825]
etichette    = ["1 anno (365 gg)", "3 anni (1095 gg)", "5 anni (1825 gg)"]
min_t, max_t = t_test.min(), t_test.max()

tempi_validi, label_validi = [], []
for t_g, lbl in zip(tempi_giorni, etichette):
    if min_t < t_g < max_t:
        tempi_validi.append(t_g)
        label_validi.append(lbl)

auc_vals, mean_auc = None, None
if tempi_validi:
    auc_vals, mean_auc = cumulative_dynamic_auc(y_train, y_test_sksurv, risk_scores, np.array(tempi_validi))

print(f"  C-index (test set) : {c_index:.4f}")
print(f"  IBS                : {ibs:.4f} (baseline ~0.25, ↓ meglio)")
if mean_auc:
    print(f"  Mean AUC           : {mean_auc:.4f}")
    for lbl, auc in zip(label_validi, auc_vals):
        print(f"    - AUC a {lbl} : {auc:.4f}")

# ==============================================================================
# FASE 2: STRATIFICAZIONE DEL RISCHIO
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 2: STRATIFICAZIONE CLINICA (Alto vs Basso Rischio)")
print("=" * 62)

median_risk = np.median(risk_scores)
mask_high   = risk_scores > median_risk
mask_low    = ~mask_high

mean_surv_high = preds_df[mask_high].mean(axis=0)
mean_surv_low  = preds_df[mask_low].mean(axis=0)
times_plot     = preds_df.columns.values.astype(float)

fig1, ax1 = plt.subplots(figsize=(10, 6))
ax1.step(times_plot, mean_surv_low,  where="post", label="Basso Rischio", color="#2ca02c", lw=2.5)
ax1.step(times_plot, mean_surv_high, where="post", label="Alto Rischio", color="#d62728", lw=2.5)
ax1.fill_between(times_plot, mean_surv_low, mean_surv_high, alpha=0.08, color="grey", step="post")
ax1.set_title("Curve di Sopravvivenza (XGBSE): Alto vs Basso Rischio", fontsize=13, fontweight="bold")
ax1.set_ylabel("Probabilità di Sopravvivenza", fontsize=11)
ax1.set_xlabel("Giorni di Follow-up", fontsize=11)
ax1.set_ylim(0, 1.02)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.35, linestyle="--")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "xgbse_survival_curves.png"), dpi=150)
plt.close()
print(f"  [+] Grafico curve salvato in: {RESULTS_DIR}")

# ==============================================================================
# FASE 3: XAI GLOBALE (PERMUTATION FEATURE IMPORTANCE)
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 3: PERMUTATION FEATURE IMPORTANCE")
print("=" * 62)
print("  Calcolo in corso (10 ripetizioni)...")

class _XGBSEWrapper:
    def fit(self, X, y=None): return self
    def predict(self, X):
        p_df = model.predict(X if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=feature_cols))
        return 1.0 - p_df.iloc[:, -1].values

def _score_fn(estimator, X, y):
    risk = estimator.predict(X)
    return concordance_index_censored(y["event"], y["time"], risk)[0]

perm = permutation_importance(
    _XGBSEWrapper(), X_test, y_test_sksurv,
    scoring=_score_fn,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

imp_mean = np.maximum(perm.importances_mean, 0)
imp_pct  = 100.0 * imp_mean / (imp_mean.sum() + 1e-12)

df_imp = pd.DataFrame({
    "Feature":        feature_cols,
    "Importanza (%)": imp_pct,
    "Std":            perm.importances_std,
}).sort_values("Importanza (%)", ascending=False)

fig2, ax2 = plt.subplots(figsize=(9, 6))
ax2.barh(df_imp["Feature"][::-1], df_imp["Importanza (%)"][::-1], color="#1f77b4", xerr=df_imp["Std"][::-1], capsize=3)
ax2.set_xlabel("Importanza relativa (%)", fontsize=11)
ax2.set_title("Permutation Feature Importance – XGBSE", fontsize=12, fontweight="bold")
ax2.grid(axis="x", alpha=0.35, linestyle="--")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "xgbse_permutation_importance.png"), dpi=150)
plt.close()
print(f"  [+] Grafico Permutation salvato in: {RESULTS_DIR}")

# ==============================================================================
# FASE 4: XAI LOCALE (SHAP)
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 4: XAI LOCALE (SHAP)")
print("=" * 62)

def _predict_risk_shap(X_raw):
    if not isinstance(X_raw, pd.DataFrame):
        X_raw = pd.DataFrame(X_raw, columns=feature_cols)
    p_df = model.predict(X_raw)
    return 1.0 - p_df.iloc[:, -1].values

background     = shap.sample(X_train_df, 100, random_state=RANDOM_STATE)
explainer_shap = shap.Explainer(_predict_risk_shap, background)

print("  Calcolo SHAP su tutto il test set...")
shap_values_all = explainer_shap(X_test)

plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_values_all, max_display=13, show=False)
plt.title("SHAP Summary – XGBSE (Contributo al Risk Score)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "xgbse_shap_summary.png"), dpi=150, bbox_inches="tight")
plt.close()
print("  [+] SHAP Summary plot salvato")

idx_alto  = int(np.argmax(risk_scores))
idx_basso = int(np.argmin(risk_scores))

for label, idx in [("ALTO RISCHIO", idx_alto), ("BASSO RISCHIO", idx_basso)]:
    plt.figure(figsize=(14, 7))
    try:
        shap.plots.waterfall(shap_values_all[idx], max_display=13, show=False, matplotlib=True)
    except TypeError:
        shap.plots.waterfall(shap_values_all[idx], max_display=13, show=False)

    ax_wf = plt.gca()
    plt.title(f"SHAP Waterfall – XGBSE – Paziente {label}", fontsize=12, fontweight="bold", pad=12)
    ax_wf.tick_params(axis="y", labelsize=9)
    for text in ax_wf.texts:
        if "f(x)" not in text.get_text() and "base value" not in text.get_text():
            text.set_fontsize(8)
    plt.subplots_adjust(left=0.42, right=0.95, top=0.90, bottom=0.10)
    ax_wf.grid(axis="x", alpha=0.3, linestyle="--")

    fname = f"xgbse_shap_waterfall_{label.lower().replace(' ', '_')}.png"
    plt.savefig(os.path.join(RESULTS_DIR, fname), dpi=150, bbox_inches="tight")
    plt.close()

print("  [+] SHAP Waterfall plots salvati per pazienti ad alto e basso rischio")

# ==============================================================================
# RIEPILOGO FINALE
# ==============================================================================
print("\n" + "=" * 62)
print("  RIEPILOGO FINALE – XGBSE (DebiasedBCE)")
print("=" * 62)
print(f"  ✅ Esecuzione completata. Tutti i grafici sono salvati in:\n  {RESULTS_DIR}")

  EVALUATION & XAI - XGBSE (DebiasedBCE)
[+] Artefatto caricato | CV C-index: 0.6636 | Feature: 13
[+] Test set: 132 pz | 56 eventi

  FASE 1: METRICHE GLOBALI
  C-index (test set) : 0.7149
  IBS                : 0.2103 (baseline ~0.25, ↓ meglio)
  Mean AUC           : 0.7262
    - AUC a 1 anno (365 gg) : 0.7759
    - AUC a 3 anni (1095 gg) : 0.6904
    - AUC a 5 anni (1825 gg) : 0.7329

  FASE 2: STRATIFICAZIONE CLINICA (Alto vs Basso Rischio)
  [+] Grafico curve salvato in: C:\Users\nicon\PycharmProjects\ProgettoTesi\XGBoostSurvival\results

  FASE 3: PERMUTATION FEATURE IMPORTANCE
  Calcolo in corso (10 ripetizioni)...
  [+] Grafico Permutation salvato in: C:\Users\nicon\PycharmProjects\ProgettoTesi\XGBoostSurvival\results

  FASE 4: XAI LOCALE (SHAP)
  Calcolo SHAP su tutto il test set...


PermutationExplainer explainer: 133it [00:12,  2.39it/s]                         


  [+] SHAP Summary plot salvato
  [+] SHAP Waterfall plots salvati per pazienti ad alto e basso rischio

  RIEPILOGO FINALE – XGBSE (DebiasedBCE)
  ✅ Esecuzione completata. Tutti i grafici sono salvati in:
  C:\Users\nicon\PycharmProjects\ProgettoTesi\XGBoostSurvival\results
